# 09 Lab — Adjustments: Defending Open Positions

This is the tool's reason to exist. We take two positions from earlier modules — a **short put** and
an **iron condor** on DEMO — replay adverse moves with `analyzer.scenario_grid` +
`viz.plot_pnl_heatmap`, then **implement** each adjustment as a new `Position`: close the old legs at
model prices with `pricing.bsm_price`, open new legs, and compare **payoff, greeks, and POP** before
vs after.

Three adjustments: (A) roll a tested short put **down-and-out for a credit**; (B) roll the **untested
call side** of a condor **down for a credit**; (C) **convert a breached** condor put side **to a
butterfly** to cap the loss. Runs offline, top-to-bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, greeks, viz, pricing
from optionslab.position import Position, OptionLeg, StockLeg
r = lambda x: round(float(x), 2)

def bank_credit(pos, credit):
    """Fold realized credit ($) from closed legs into basis for campaign analysis."""
    legs = list(pos.legs)
    for i, l in enumerate(legs):
        if isinstance(l, OptionLeg) and l.quantity < 0:
            bumped = l.premium + credit / (abs(l.quantity) * l.multiplier)
            legs[i] = OptionLeg(l.kind, l.strike, l.expiry, l.quantity, bumped)
            break
    return Position(tuple(legs), pos.label + " (campaign basis)")

## Scenario A — a tested short put

You sold the DEMO 95 put at 45 DTE for 1.58 (a cash-secured put; you were mildly bullish and happy
to own at 95). DEMO has since fallen. First, replay the damage: a scenario grid of P&L across lower
spots and elapsed days.

In [ ]:
sp = strategies.cash_secured_put((95, 1.58), expiry=45/365)
grid = analyzer.scenario_grid(sp, spots=[88, 91, 94, 97, 100], days_forward=[0, 10, 20], vol=0.30)
ax = viz.plot_pnl_heatmap(grid)
ax.set_title("Short 95 put: P&L as DEMO falls (adverse move)")

The red lower-left corner is the problem: a drop toward 88 with time elapsed is a growing
loss. Define the **current** adverse state — spot 92, 20 days elapsed (25 DTE left), IV spiked to
0.32 — and check whether the **thesis is intact**. If you would still be happy to own DEMO lower,
adjust; if the reason you sold the put is gone, close. Here we assume the thesis holds.

In [ ]:
spot, rem, vol = 92.0, 25/365, 0.32
buyback = pricing.bsm_price("put", spot, 95, rem, vol)   # close old leg at model price
realized = (buyback - 1.58) * (-1) * 100                 # short leg: (close-entry)*qty*mult
orig_now = strategies.cash_secured_put((95, 1.58), expiry=rem)   # the trade as it stands, 25 DTE
print("buyback 95p:", r(buyback), "| realized P&L on close $:", r(realized))
print("orig POP@92:", r(analyzer.probability_of_profit(orig_now, spot, vol)),
      "| delta $:", r(greeks.position_greeks(orig_now, spot, vol).delta))

The short put is down ~\$327 and carries **+63 deltas** (it behaves increasingly like long
stock as it goes ITM). **Roll down-and-out for a credit:** buy back the 95 put, sell the **90** put
in a later cycle (120 DTE). Rolling far enough out funds the strike move down and still collects
cash — the roll-for-credit rule.

In [ ]:
new_prem = pricing.bsm_price("put", spot, 90, 120/365, vol)   # open new leg at model price
roll_net = (new_prem - buyback) * 100
adj = strategies.cash_secured_put((90, round(new_prem, 2)), expiry=120/365)
print("sell new 90p:", r(new_prem), "| roll net $ (credit if > 0):", r(roll_net))
print("adj POP@92:", r(analyzer.probability_of_profit(adj, spot, vol)),
      "| delta $:", r(greeks.position_greeks(adj, spot, vol).delta),
      "| max_loss $:", r(analyzer.max_loss(adj)))

The roll is a **net credit**, lowers the obligation strike 95 → 90, cuts delta 63 → 41, and
lifts forward POP 0.41 → 0.66. The ~\$327 already lost is **sunk**; the decision is whether the *new*
trade is good — and it is. Compare the two payoff curves (the new one is shifted by the realized
loss you carry into it).

In [ ]:
spots = np.linspace(80, 105, 121)
before = payoff.pnl_curve(orig_now, spots)
after = payoff.pnl_curve(adj, spots) + realized      # campaign incl. sunk roll loss
fig, ax = plt.subplots()
ax.plot(spots, before, label="do nothing (short 95p, 25 DTE)")
ax.plot(spots, after, label="rolled to 90p 120 DTE (+ sunk loss)")
ax.axhline(0, color="k", lw=.7); ax.axvline(92, color="grey", lw=.6)
ax.legend(); ax.set_title("Scenario A: before vs after the roll")

## Scenario B — condor, put side tested: roll the *untested* call side down

The flagship condor from module 05: DEMO iron condor 87.5 / 92.5 / 107.5 / 112.5 at 45 DTE, \$140
credit. DEMO has slipped to 93 (25 DTE left, IV 0.30) — the **put** side is tested. First replay
it, then defend by rolling the **safe call side down** for a credit.

In [ ]:
ic = strategies.iron_condor((87.5,0.37),(92.5,1.01),(107.5,1.19),(112.5,0.43), expiry=25/365)
grid = analyzer.scenario_grid(ic, spots=[89,92,95,100,105], days_forward=[0,10,20], vol=0.30)
ax = viz.plot_pnl_heatmap(grid); ax.set_title("Iron condor: put side tested as DEMO falls to 93")

In [ ]:
spot, rem, vol = 93.0, 25/365, 0.30
# close old call spread at model prices; open a lower one 102.5/107.5
bb = pricing.bsm_price("call", spot, 107.5, rem, vol); sl = pricing.bsm_price("call", spot, 112.5, rem, vol)
old_call_realized = (1.19-0.43)*100 - (bb-sl)*100        # banked call credit minus cost to close
new_sc = pricing.bsm_price("call", spot, 102.5, rem, vol); new_lc = pricing.bsm_price("call", spot, 107.5, rem, vol)
roll_net = (new_sc-new_lc)*100 - (bb-sl)*100
print("close old call spread realized $:", r(old_call_realized), "| roll net $:", r(roll_net))

In [ ]:
adj = strategies.iron_condor((87.5,0.37),(92.5,1.01),
        (102.5, round(new_sc,2)), (107.5, round(new_lc,2)), expiry=rem)
adjc = bank_credit(adj, old_call_realized)               # fold banked cash into basis
for name, p in [("orig", ic), ("adjusted", adjc)]:
    s = analyzer.summarize(p, spot, vol)
    print(f"{name:9} credit={-s['net_premium']:6.0f} BEs={[r(x) for x in s['breakevens']]} "
          f"POP={s['probability_of_profit']:.2f} maxL={r(s['max_loss'])}")

Reading the numbers: the roll banks another ~\$22 (campaign credit \$140 → \$162), which
**lowers the tested-side breakeven 91.10 → 90.88** (more downside room), **cuts max loss \$360 →
\$338**, and flattens delta. The price you pay is upside room — the upper breakeven drops 108.9 →
104.1 and POP dips slightly. That is the trade: harvest the safe side's profit to defend the tested
side, accepting a narrower top. Roll the untested side in **only** as long as it keeps paying and
does not become the new problem.

## Scenario C — put side *breached*: convert to a butterfly to cap the loss

DEMO keeps falling to 92 (20 DTE, IV 0.31): the short 92.5 put is breached and rolling for a credit
is gone. **Convert the breached short put spread into a long put butterfly** by selling one more 92.5
put and buying a 97.5 put — turning the runaway short into the fly's body. This costs a debit (the
allowed exception to the credit rule): you are *buying a cap on the loss*.

In [ ]:
spot, rem, vol = 92.0, 20/365, 0.31
put_spread = strategies.custom(OptionLeg("put",87.5,rem,1,0.37), OptionLeg("put",92.5,rem,-1,1.01),
                               label="breached put spread")
add_sell = pricing.bsm_price("put", spot, 92.5, rem, vol)   # sell one more body put
add_buy = pricing.bsm_price("put", spot, 97.5, rem, vol)    # buy the upper wing
print("add legs: sell 92.5p", r(add_sell), "buy 97.5p", r(add_buy),
      "| net debit $:", r((add_buy-add_sell)*100))

In [ ]:
fly = strategies.custom(
    OptionLeg("put", 87.5, rem, 1, 0.37),
    OptionLeg("put", 92.5, rem, -2, round((1.01+add_sell)/2, 2)),   # both short 92.5 puts
    OptionLeg("put", 97.5, rem, 1, round(add_buy, 2)),
    label="converted put fly 87.5/92.5/97.5")
for name, p in [("breached spread", put_spread), ("converted fly", fly)]:
    print(f"{name:16} maxL={r(analyzer.max_loss(p)):>8} maxP={r(analyzer.max_profit(p)):>7} "
          f"POP@92={analyzer.probability_of_profit(p, spot, vol):.2f}")

In [ ]:
spots = np.linspace(82, 100, 121)
fig, ax = plt.subplots()
ax.plot(spots, payoff.pnl_curve(put_spread, spots), label="breached spread (uncapped side)")
ax.plot(spots, payoff.pnl_curve(fly, spots), label="converted to put fly (capped)")
ax.axhline(0, color="k", lw=.7); ax.axvline(92, color="grey", lw=.6)
ax.legend(); ax.set_title("Scenario C: convert breached side to a fly caps the loss")

The conversion cuts max loss from ~\$436 to ~\$276 **and** builds a ~\$221 profit tent if
DEMO stabilizes near 92.5. POP falls (the fly only wins in a narrow band) — that is the honest
trade-off: you pay a debit to *cap catastrophe* and buy a recovery zone, not to raise your odds.

## Delta hedging and the 21-DTE decision

The tested short put carried **+63 deltas** — it drifts long as it goes ITM. If your view is still
range-bound rather than directional, **hedge the delta** by shorting shares (1 delta each) to bring
the book back toward flat, instead of rolling.

In [ ]:
d = greeks.position_greeks(orig_now, 92, 0.32).delta
hedged = strategies.custom(OptionLeg("put",95,25/365,-1,1.58), StockLeg(-round(d), 92.0),
                           label="short put + stock hedge")
print("unhedged delta $:", r(d), "-> hedged delta $:", r(greeks.position_greeks(hedged, 92, 0.32).delta))

And the **21-DTE rule**: short-premium positions carry negative gamma that turns violent in
the final weeks. At ~21 DTE you *decide* — take the winner, roll the whole thing to a new cycle for a
credit, or close a broken loser. You do not carry short gamma into the last week and hope.

## Experiments

1. Scenario A: try rolling to the **92.5** put at 120 DTE instead of 90. More credit, but does it
   lower your obligation strike enough? Re-check `roll_net`, POP, and delta.
2. Scenario A: make the move worse — set `spot=88`, `vol=0.36`. Does a credit roll still exist, or
   is this now a *close* (thesis-broken) situation?
3. Scenario B: keep rolling the untested call side to 100/105. Watch the upper breakeven collapse —
   at what point does the "safe" side become the new tested side?
4. Scenario C: convert **earlier**, at spot 92.5 with 25 DTE. Is the debit smaller? Does the fly's
   tent sit better relative to price?
5. Delta hedge with a **put** instead of stock: buy a cheap OTM put to add negative delta. Compare
   the effect on gamma and theta vs the stock hedge (`position_greeks` with `which` greeks).